# Notebook 8: Sensitivity Analysis

## 왜 이 노트북이 필요한가?

현재 3개 핵심 하이퍼파라미터가 근거 없이 고정돼 있다:
1. IF `contamination=0.15` — precision/recall을 결정
2. PCA `n_components=0.95` (가변) — 실제 차원 컨트롤 불가
3. HMM+IF α=0.7 — NB04 grid search가 test set에 overfit될 수 있음

리뷰어는 "결과가 이 파라미터에 민감하다면 방법론이 robust하지 않다"고 reject한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

## 0. 공통 데이터 로드

In [ ]:
import numpy as np
import pandas as pd
import ast, json, pickle
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']

benchmark_df = pd.read_csv(f'{LABELS_DIR}/synthetic_anomaly_benchmark.csv')
benchmark_df['sequence'] = benchmark_df['sequence'].apply(ast.literal_eval)
benchmark_labels = benchmark_df['is_anomaly'].values

with open(f'{MODELS_DIR}/hmm_model.pkl', 'rb') as f:
    hmm_model = pickle.load(f)
with open(f'{MODELS_DIR}/hmm_thresholds.json') as f:
    thresholds = json.load(f)
with open(f'{MODELS_DIR}/pca_model.pkl', 'rb') as f:
    pca_bundle = pickle.load(f)

embeddings_raw = np.load(f'{LABELS_DIR}/embeddings.npy')     # (N, 1536)
embeddings_pca = np.load(f'{LABELS_DIR}/embeddings_pca.npy') # (N, k)

df_train = pd.read_csv(f'{LABELS_DIR}/weak_labels.csv')

print(f'원본 임베딩: {embeddings_raw.shape}')
print(f'PCA 임베딩: {embeddings_pca.shape}')
print(f'벤치마크: {len(benchmark_df)}개')

## 1. IF contamination Sensitivity

In [ ]:
CONTAMINATION_RANGE = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

# 벤치마크 샘플에 대한 PCA 임베딩 추출 (original_deck_id 기반)
def get_bench_emb(benchmark_df, df_train, embeddings_pca):
    """벤치마크 각 행의 original_deck_id 슬라이드 PCA 임베딩 평균 반환."""
    emb_map = {}  # deck_id → 행 인덱스 배열
    for deck_id, group in df_train.groupby('deck_id'):
        emb_map[deck_id] = group.index.tolist()

    bench_embs = []
    for _, row in benchmark_df.iterrows():
        idxs = emb_map.get(row['original_deck_id'], [])
        if not idxs:
            bench_embs.append(np.zeros(embeddings_pca.shape[1]))
        else:
            seq_len = len(row['sequence'])
            sel = idxs[:seq_len]
            bench_embs.append(embeddings_pca[sel].mean(axis=0))
    return np.array(bench_embs)

bench_emb_pca = get_bench_emb(benchmark_df, df_train, embeddings_pca)

contamination_results = []
for cont in CONTAMINATION_RANGE:
    iso = IsolationForest(n_estimators=200, contamination=cont, random_state=42)
    iso.fit(embeddings_pca)
    raw = iso.decision_function(bench_emb_pca)
    s_min, s_max = raw.min(), raw.max()
    scores = 1.0 - (raw - s_min) / (s_max - s_min + 1e-8)
    auc = roc_auc_score(benchmark_labels, scores)
    contamination_results.append({'contamination': cont, 'auc': float(auc)})
    print(f'contamination={cont:.2f} | AUC={auc:.4f}')

auc_range_cont = max(r['auc'] for r in contamination_results) - min(r['auc'] for r in contamination_results)
print(f'\nAUC 변화 폭: {auc_range_cont:.4f}')
print('⚠ 민감' if auc_range_cont > 0.05 else '✓ 안정적')

## 2. PCA 차원 수 Sensitivity

In [ ]:
pca_dims = [16, 32, 64, 128, 256, 512]
scaler = StandardScaler()
emb_scaled = scaler.fit_transform(embeddings_raw)

pca_results = []
for n_dim in pca_dims:
    pca = PCA(n_components=n_dim, random_state=42)
    emb_reduced = pca.fit_transform(emb_scaled)
    variance_ratio = pca.explained_variance_ratio_.sum()

    # 벤치마크 임베딩 변환
    bench_emb_nd = []
    emb_map = {}
    for deck_id, group in df_train.groupby('deck_id'):
        emb_map[deck_id] = group.index.tolist()
    for _, row in benchmark_df.iterrows():
        idxs = emb_map.get(row['original_deck_id'], [])
        if not idxs:
            bench_emb_nd.append(np.zeros(n_dim))
        else:
            sel = idxs[:len(row['sequence'])]
            bench_emb_nd.append(emb_reduced[sel].mean(axis=0))
    bench_emb_nd = np.array(bench_emb_nd)

    iso = IsolationForest(n_estimators=200, contamination=0.15, random_state=42)
    iso.fit(emb_reduced)
    raw = iso.decision_function(bench_emb_nd)
    s_min, s_max = raw.min(), raw.max()
    scores = 1.0 - (raw - s_min) / (s_max - s_min + 1e-8)
    auc = roc_auc_score(benchmark_labels, scores)

    pca_results.append({'n_components': n_dim, 'variance_explained': float(variance_ratio), 'auc': float(auc)})
    print(f'PCA dim={n_dim:4d} | var={variance_ratio:.3%} | AUC={auc:.4f}')

optimal_dim = pca_dims[np.argmax([r['auc'] for r in pca_results])]
current_dim = embeddings_pca.shape[1]
print(f'\n최적 고정 차원: {optimal_dim} | 현재 95%var 차원: {current_dim}')

## 3. Alpha 가중치 — Train/Test 분리 Overfit 검증

In [ ]:
# 올바른 방법: train에서 탐색 → 독립 test에서 1회 평가
np.random.seed(42)
perm = np.random.permutation(len(benchmark_df))
split = int(len(benchmark_df) * 0.7)
train_idx = perm[:split]
test_idx   = perm[split:]

# HMM 스코어 계산
all_hmm_scores = []
for _, row in benchmark_df.iterrows():
    seq = np.array(row['sequence']).reshape(-1, 1)
    if len(seq) < 2:
        all_hmm_scores.append(0.5); continue
    ll = hmm_model.score(seq) / len(seq)
    z  = (thresholds['mean'] - ll) / (thresholds['std'] + 1e-8)
    all_hmm_scores.append(float(np.clip(z / 3.0, 0, 1)))

# IF 스코어
iso_eval = IsolationForest(n_estimators=200, contamination=0.15, random_state=42)
iso_eval.fit(embeddings_pca)
raw = iso_eval.decision_function(bench_emb_pca)
s_min, s_max = raw.min(), raw.max()
all_if_scores = (1.0 - (raw - s_min) / (s_max - s_min + 1e-8)).tolist()

alphas = np.linspace(0.0, 1.0, 21)

# Train split으로 alpha 탐색
train_hmm = np.array(all_hmm_scores)[train_idx]
train_if  = np.array(all_if_scores)[train_idx]
train_lbl = benchmark_labels[train_idx]

train_aucs = []
for alpha in alphas:
    combined = alpha * train_if + (1 - alpha) * train_hmm
    if len(np.unique(train_lbl)) < 2: train_aucs.append(0.5); continue
    train_aucs.append(roc_auc_score(train_lbl, combined))

best_alpha_train = alphas[np.argmax(train_aucs)]
print(f'Train 최적 α: {best_alpha_train:.2f}')

# Test split에서 1회만 평가
test_hmm = np.array(all_hmm_scores)[test_idx]
test_if  = np.array(all_if_scores)[test_idx]
test_lbl = benchmark_labels[test_idx]
test_combined = best_alpha_train * test_if + (1 - best_alpha_train) * test_hmm
auc_test_proper = roc_auc_score(test_lbl, test_combined)

# NB04 방식(전체 데이터 탐색)과 비교
all_aucs_full = []
for alpha in alphas:
    combined = alpha * np.array(all_if_scores) + (1 - alpha) * np.array(all_hmm_scores)
    all_aucs_full.append(roc_auc_score(benchmark_labels, combined))
best_auc_full = max(all_aucs_full)

overfit_gap = best_auc_full - auc_test_proper
print(f'Test AUC (proper split): {auc_test_proper:.4f}')
print(f'Full-data best AUC:      {best_auc_full:.4f}')
print(f'Overfit gap:             {overfit_gap:.4f}')
if overfit_gap > 0.02:
    print('⚠ NB04 alpha grid search가 test set에 overfit됨 — 논문에서 proper split 사용 필요')

## 4. Sensitivity 결과 저장 및 시각화

In [ ]:
sensitivity_results = {
    'contamination': {
        'values':   CONTAMINATION_RANGE,
        'aucs':     [r['auc'] for r in contamination_results],
        'range':    float(auc_range_cont),
        'sensitive': bool(auc_range_cont > 0.05),
        'selected': 0.15,
    },
    'pca_dim': {
        'values':    pca_dims,
        'aucs':      [r['auc'] for r in pca_results],
        'optimal':   int(optimal_dim),
        'current':   int(current_dim),
        'mismatch':  bool(optimal_dim != current_dim),
    },
    'alpha': {
        'best_alpha_train': float(best_alpha_train),
        'auc_test_proper':  float(auc_test_proper),
        'auc_full_best':    float(best_auc_full),
        'overfit_gap':      float(overfit_gap),
        'overfit_warning':  bool(overfit_gap > 0.02),
    },
}
with open(f'{MODELS_DIR}/sensitivity_results.json', 'w') as f:
    json.dump(sensitivity_results, f, indent=2)
print(json.dumps(sensitivity_results, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# contamination
cont_aucs = [r['auc'] for r in contamination_results]
axes[0].plot(CONTAMINATION_RANGE, cont_aucs, 'o-', color='steelblue')
axes[0].axvline(0.15, color='red', linestyle='--', label='현재 0.15')
axes[0].fill_between(CONTAMINATION_RANGE,
                     [min(cont_aucs)]*len(CONTAMINATION_RANGE),
                     cont_aucs, alpha=0.1, color='steelblue')
axes[0].set_xlabel('contamination'); axes[0].set_ylabel('AUC')
axes[0].set_title(f'IF contamination (range={auc_range_cont:.3f})')
axes[0].legend()

# PCA dim
dim_aucs = [r['auc'] for r in pca_results]
axes[1].plot(pca_dims, dim_aucs, 's-', color='green')
axes[1].axvline(optimal_dim, color='green', linestyle='--', label=f'최적={optimal_dim}')
axes[1].axvline(current_dim, color='red', linestyle=':', label=f'현재={current_dim}')
axes[1].set_xlabel('PCA n_components'); axes[1].set_title('PCA 차원 sensitivity')
axes[1].legend()

# alpha
axes[2].plot(alphas, train_aucs, 'o-', label='Train grid search', color='tomato')
axes[2].axvline(best_alpha_train, color='red', linestyle='--')
axes[2].axhline(auc_test_proper, color='blue', linestyle='--',
                label=f'Test proper={auc_test_proper:.3f}')
axes[2].set_xlabel('α (IF weight)'); axes[2].set_title(f'Alpha overfit (gap={overfit_gap:.3f})')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/sensitivity_analysis.png', dpi=120)
plt.show()
print('저장: sensitivity_analysis.png')

In [ ]:
print('=== Notebook 8 완료 ===')
print(f'Sensitivity 결과: {MODELS_DIR}/sensitivity_results.json')
print()
print('요약:')
print(f'  contamination 민감도: {"⚠ 민감" if auc_range_cont > 0.05 else "✓ 안정적"}')
print(f'  최적 PCA 차원: {optimal_dim} (현재 {current_dim}){" ← 수정 권장" if optimal_dim != current_dim else ""}')
print(f'  Alpha overfit gap: {overfit_gap:.4f}{" ← 논문에서 proper split 사용" if overfit_gap > 0.02 else " ← 허용 범위"}')
print('\nNotebook 9 (Stats Significance)으로 이동하세요.')